In [1]:
!pip install sounddevice edge-tts sarvamai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.1/229.1 kB 13.4 MB/s eta 0:00:00


In [2]:
!apt-get update && apt-get install -y libportaudio2 libportaudio-dev

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,652 kB]
Get:13 http://security.ubuntu.com/ubuntu j

In [3]:
! pip install asyncio


In [ ]:
!apt-get update && apt-get install -y portaudio19-dev
!pip install PyAudio
!pip install genai

import asyncio
import base64
import pyaudio
from genai import Client # New SDK
from sarvamai import AsyncSarvamAI

# --- API Configuration ---
import os
from dotenv import load_dotenv

load_dotenv()

SARVAM_API_KEY = os.environ["SARVAM_API_KEY"]
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]


# Initialize Clients
gemini_client = Client(api_key=GEMINI_API_KEY)
sarvam_client = AsyncSarvamAI(api_subscription_key=SARVAM_API_KEY)

SYSTEM_PROMPT = """
You are a 112 Emergency Dispatcher.
Speak ONLY in Hindi/Hinglish.
Ask for: Location, Incident Type, Injuries.
Keep responses under 10 words. Direct and urgent.
"""

# Audio Constants
RATE_STT = 16000
RATE_TTS = 24000 # Sarvam Bulbul v3 default
CHUNK = 1024

p = pyaudio.PyAudio()

async def run_emergency_agent():
    # Setup Mic (Input)
    mic = p.open(format=pyaudio.pa16, channels=1, rate=RATE_STT,
                 input=True, frames_per_buffer=CHUNK)

    # Setup Speaker (Output)
    speaker = p.open(format=pyaudio.paInt16, channels=1, rate=RATE_TTS,
                    output=True)

    print("--- 112 EMERGENCY LINE ACTIVE ---")

    async with sarvam_client.speech_to_text_streaming.connect(
        model="saaras:v3", language_code="hi-IN", mode="codemix"
    ) as stt:

        while True:
            # 1. Capture Mic Audio
            try:
                raw_mic_data = mic.read(CHUNK, exception_on_overflow=False)
                await stt.transcribe(audio=base64.b64encode(raw_mic_data).decode('utf-8'))

                # 2. Get STT Result
                res = await stt.recv()
                if res.get("transcript"):
                    user_text = res["transcript"]
                    print(f"Caller: {user_text}")

                    # 3. Gemini Reasoning (New SDK Syntax)
                    response = gemini_client.models.generate_content(
                        model="gemini-2.0-flash", # Use the 2.0 version for lowest latency
                        contents=f"{SYSTEM_PROMPT}\nUser says: {user_text}"
                    )
                    ai_text = response.text
                    print(f"AI: {ai_text}")

                    # 4. Sarvam TTS
                    tts_res = await sarvam_client.text_to_speech.convert(
                        model="bulbul:v3",
                        text=ai_text,
                        target_language_code="hi-IN",
                        speaker="shubh"
                    )

                    # 5. Fixed Audio Playback
                    if tts_res.audios:
                        audio_bytes = base64.b64decode(tts_res.audios[0])
                        # Write to speaker stream
                        speaker.write(audio_bytes)

            except Exception as e:
                print(f"Streaming error: {e}")
                continue

if __name__ == "__main__":
    try:
        asyncio.run(run_emergency_agent())
    except KeyboardInterrupt:
        print("\nClosing...")
        p.terminate()

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

ModuleNotFoundError: No module named 'genai'

In [ ]:
 pip install PortAudio

ERROR: Could not find a version that satisfies the requirement PortAudio (from versions: none)
ERROR: No matching distribution found for PortAudio
